In [1]:
library(keras)
library(tensorflow)
library(tidyverse)
library(recipes)
set.seed(2) 

Warning message:
"le package 'keras' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tensorflow' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyverse' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'ggplot2' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tibble' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'readr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'forcats' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'lubridate' a été compilé avec la version R 4.2.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.2     ✔ tibble    3.2.1
✔ lubridate 1.9.2    

In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [3]:
# data<-read.csv("train_values.csv",stringsAsFactors = T)
# data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
# datam<-merge(data,data_labels,by=c('building_id','building_id'))

In [6]:
# target_variable<-match('damage_grade', colnames(datam))
# id_loc_1 <- match('geo_level_1_id', colnames(datam))
# id_loc_2 <- match('geo_level_2_id', colnames(datam))
# id_loc_3 <- match('geo_level_3_id', colnames(datam))
# id_loc_1_vals<-sort(unique(datam[,id_loc_1]))
# id_loc_2_vals<-sort(unique(datam[,id_loc_2]))
# id_loc_3_vals<-sort(unique(datam[,id_loc_3]))
# id_loc_1_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_1_vals)))
# colnames(id_loc_1_df)<-c('id','mean')
# id_loc_2_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_2_vals)))
# colnames(id_loc_2_df)<-c('id','mean')
# id_loc_3_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_3_vals)))
# colnames(id_loc_3_df)<-c('id','mean')

# for (i in 1:length(id_loc_1_vals)){
#     id_loc_1_df[i,1]<-id_loc_1_vals[i]
#     id_loc_1_df[i,2]<-mean(filter(datam,geo_level_1_id == id_loc_1_vals[i])[,target_variable])
# }
# for (i in 1:length(id_loc_2_vals)){
#     id_loc_2_df[i,1]<-id_loc_2_vals[i]
#     id_loc_2_df[i,2]<-mean(filter(datam,geo_level_2_id == id_loc_2_vals[i])[,target_variable])
# }
# for (i in 1:length(id_loc_3_vals)){
#     id_loc_3_df[i,1]<-id_loc_3_vals[i]
#     id_loc_3_df[i,2]<-mean(filter(datam,geo_level_3_id == id_loc_3_vals[i])[,target_variable])
# }

In [11]:
# datam<-add_column(datam,geo_level_1_mean_damage = NA,.after='geo_level_1_id')
# datam<-add_column(datam,geo_level_2_mean_damage = NA,.after='geo_level_2_id')
# datam<-add_column(datam,geo_level_3_mean_damage = NA,.after='geo_level_3_id')
# id_loc_1 <- match('geo_level_1_id', colnames(datam))
# id_loc_2 <- match('geo_level_2_id', colnames(datam))
# id_loc_3 <- match('geo_level_3_id', colnames(datam))
# for (i in 1:nrow(datam)){
#     datam[i,id_loc_1+1]<-filter(id_loc_1_df,id == datam[i,id_loc_1])[1,2]
#     datam[i,id_loc_2+1]<-filter(id_loc_2_df,id == datam[i,id_loc_2])[1,2]
#     datam[i,id_loc_3+1]<-filter(id_loc_3_df,id == datam[i,id_loc_3])[1,2]
# }
#write.csv(datam,'data_geoprocessed.csv')


dataNN<-read.csv("data_target_encoding_NN.csv",stringsAsFactors = T)
id_variable <- match('building_id', colnames(dataNN))
target_indices<-which(grepl('damage_grade',colnames(dataNN)))

#DO NOT FORGET processing for test values which zones could not be in training set


In [4]:
summary(datam)

       X           building_id      geo_level_1_id geo_level_1_mean_damage
 Min.   :     1   Min.   :      4   Min.   : 0.0   Min.   :1.731          
 1st Qu.: 65151   1st Qu.: 261190   1st Qu.: 7.0   1st Qu.:2.026          
 Median :130301   Median : 525757   Median :12.0   Median :2.172          
 Mean   :130301   Mean   : 525676   Mean   :13.9   Mean   :2.238          
 3rd Qu.:195451   3rd Qu.: 789762   3rd Qu.:21.0   3rd Qu.:2.446          
 Max.   :260601   Max.   :1052934   Max.   :30.0   Max.   :2.794          
                                                                          
 geo_level_2_id   geo_level_2_mean_damage geo_level_3_id 
 Min.   :   0.0   Min.   :1.000           Min.   :    0  
 1st Qu.: 350.0   1st Qu.:2.018           1st Qu.: 3073  
 Median : 702.0   Median :2.210           Median : 6270  
 Mean   : 701.1   Mean   :2.238           Mean   : 6258  
 3rd Qu.:1050.0   3rd Qu.:2.480           3rd Qu.: 9412  
 Max.   :1427.0   Max.   :3.000           Max.   :12

I try without removing nzv values. We will try it later to see if it improves predict.

In [8]:
dataNN <- recipe(damage_grade ~ ., datam) %>%
  #step_nzv(everything(), -damage_grade) %>%
  step_normalize(count_floors_pre_eq, age, area_percentage, height_percentage) %>%
  step_num2factor(damage_grade,levels=c('1','2','3')) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  step_rm(geo_level_1_id,geo_level_2_id,geo_level_3_id,X,building_id) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL)

step_normalize (normalize_c1pFX): same number of columns

step_num2factor (num2factor_8OXka): same number of columns

step_dummy (dummy_hqCql): 
 new (41): land_surface_condition_n, land_surface_condition_o, ...
 removed (9): land_surface_condition, foundation_type, roof_type, ...

step_rm (rm_uoffF): 
 removed (5): X, building_id, geo_level_1_id, geo_level_2_id, ...



In [11]:
nrows<-nrow(dataNN)
split <- floor(nrows*0.8)
dataNN_idx <- sample(1:nrows)
train_data <- dataNN[dataNN_idx[1:split],]
test_data <- dataNN[dataNN_idx[(split+1):nrows],]
target_indices<-which(grepl('damage_grade',colnames(dataNN)))

In [12]:
normalizer<-layer_normalization(axis = -1L)  %>%  adapt(as.matrix(train_data[,-target_indices]))


We use categorical_crossentropy as loss function since this corresponds to our multioutput classification program

In [15]:
neuralmodel <- keras_model_sequential() %>% 
normalizer  %>% 
layer_dense(64, activation = 'relu') %>%
layer_dense(64, activation = 'relu') %>%
layer_dense(3,activation='softmax')

neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.001),
    metrics=c('accuracy')
  )
neuralmodel

Model: "sequential_2"
________________________________________________________________________________
 Layer (type)                  Output Shape               Param #    Trainable  
 normalization_1 (Normalizatio  (None, 71)                143        Y          
 n)                                                                             
 dense_7 (Dense)               (None, 64)                 4608       Y          
 dense_6 (Dense)               (None, 64)                 4160       Y          
 dense_5 (Dense)               (None, 3)                  195        Y          
Total params: 9,106
Trainable params: 8,963
Non-trainable params: 143
________________________________________________________________________________

In [16]:
yhat <- neuralmodel %>% fit(
  as.matrix(train_data[,-target_indices]),
  as.matrix(train_data[,target_indices]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 100
)
yhat


Final epoch (plot to see history):
        loss: 0.4852
    accuracy: 0.7759
    val_loss: 0.5816
val_accuracy: 0.7466 

In [13]:
yhatt<-neuralmodel  %>% evaluate(
    as.matrix(test_data[,-target_indices]),
    as.matrix(test_data[,target_indices]),
)
yhatt   # this was with old variables

loss  accuracy 
0.5943348 0.7434239

In [17]:
yhatt<-neuralmodel  %>% evaluate(
    as.matrix(test_data[,-target_indices]),
    as.matrix(test_data[,target_indices]),
)
yhatt   # this is with target encoding

loss  accuracy 
0.5780700 0.7472754

In [18]:
yhattt <- predict(neuralmodel, as.matrix(test_data[-target_indices]))
yhattt

1.334853e-05,7.855563e-02,9.214311e-01
1.644487e-02,9.721357e-01,1.141938e-02
3.440760e-02,8.116365e-01,1.539559e-01
1.892376e-07,7.617473e-01,2.382525e-01
3.664064e-04,5.950868e-01,4.045469e-01
9.569246e-06,5.713631e-01,4.286274e-01
2.483511e-09,9.999813e-01,1.874831e-05
6.127841e-04,6.073133e-01,3.920739e-01
6.037734e-01,3.857036e-01,1.052303e-02
1.862911e-03,5.401218e-01,4.580152e-01
2.075230e-04,2.673935e-01,7.323990e-01


In [19]:
yhat_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhattt)))
y_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhattt)))
for (i in 1:nrow(yhattt)){
    yhat_f[i,]<-which.max(yhattt[i,])
    y_f[i,]<-which.max(test_data[i,target_indices])  
}
colnames(yhat_f)<-'damage_grade'
colnames(y_f)<-'damage_grade'

In [16]:
print(paste('F1 Score Micro: ',F1_Score_micro(as.factor(y_f[,]),as.factor(yhat_f[,])))) #old variables

[1] "F1 Score Micro:  0.743423955795169"


In [ ]:
print(paste('F1 Score Micro: ',F1_Score_micro(as.factor(y_f[,]),as.factor(yhat_f[,])))) #target encoding

In [17]:
# data_t<-read.csv("test_values.csv",stringsAsFactors = T)

# data_t<-add_column(data_t,geo_level_1_mean_damage = NA,.after='geo_level_1_id')
# data_t<-add_column(data_t,geo_level_2_mean_damage = NA,.after='geo_level_2_id')
# data_t<-add_column(data_t,geo_level_3_mean_damage = NA,.after='geo_level_3_id')
# id_loc_1 <- match('geo_level_1_id', colnames(data_t))
# id_loc_2 <- match('geo_level_2_id', colnames(data_t))
# id_loc_3 <- match('geo_level_3_id', colnames(data_t))
# for (i in 1:nrow(data_t)){
#     data_t[i,id_loc_1+1]<-filter(id_loc_1_df,id == data_t[i,id_loc_1])[1,2]
#     data_t[i,id_loc_2+1]<-filter(id_loc_2_df,id == data_t[i,id_loc_2])[1,2]
#     data_t[i,id_loc_3+1]<-filter(id_loc_3_df,id == data_t[i,id_loc_3])[1,2]
# }
# write.csv(data_t,'testdata_geoprocessed.csv')

data_t<-read.csv("testdata_geoprocessed.csv",stringsAsFactors = T)

In [19]:
dataNN_test <- recipe( ~ ., data_t) %>%
  #step_nzv(everything(), ) %>%
  step_normalize(count_floors_pre_eq, age, area_percentage, height_percentage,building_id) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  step_rm(geo_level_1_id,geo_level_2_id,geo_level_3_id,X,building_id) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL)

step_normalize (normalize_9V4Fq): same number of columns

step_dummy (dummy_cY7sP): 
 new (38): land_surface_condition_n, land_surface_condition_o, ...
 removed (8): land_surface_condition, foundation_type, roof_type, ...

step_rm (rm_faZoC): 
 removed (4): X, geo_level_1_id, geo_level_2_id, geo_level_3_id



In [23]:
test_predict <- predict(neuralmodel, as.matrix(dataNN_test[,-1]))

In [42]:
test_predict_f<-data.frame(matrix(0,ncol = 2, nrow = nrow(test_predict)))
colnames(test_predict_f)<-c('building_id','damage_grade')
test_predict<-replace_na(test_predict,0)

for (i in 1:nrow(test_predict)){
    test_predict_f[i,1]<-data_t[i,2]
    test_predict_f[i,2]<-which.max(test_predict[i,])
}

In [45]:
write.csv(test_predict_f,'prediction_temp_NN.csv',col.names=TRUE,row.names=FALSE)

Warning message in write.csv(test_predict_f, "prediction_temp_NN.csv", col.names = TRUE, :
"une tentative de modification de 'col.names' a échoué"


In [4]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
data_old<-merge(data,data_labels,by=c('building_id','building_id'))
data_old<-filter(data_old,age<995)
dataNN<-cbind(dataNN,data_old[,c(2,3,4)])

k-fold CV: 

In [5]:
k = 5

nrows<-nrow(dataNN)
split <- floor(nrows*0.8)

# 1. Shuffle the dataset randomly.
dataNN_idx <- sample(1:nrows)


# 2. Split the dataset into k groups
max <- ceiling(nrow(dataNN)/k)
splits <- split(dataNN_idx, ceiling(seq_along(dataNN_idx)/max))


target_indices<-which(grepl('damage_grade',colnames(dataNN)))
accuracy_vec <- array(0,k)


pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

  #3.1 Take the group as a hold out or test data set
  test_data <- dataNN[splits[[i]],]


  #3.2 Take the remaining groups as a training data set
  train_data <- dataNN[-splits[[i]],]   

  normalizer<-layer_normalization(axis = -1L)  %>%  adapt(as.matrix(train_data[,-target_indices]))

  neuralmodel <- keras_model_sequential() %>% 
  normalizer  %>% 
  layer_dense(40, activation = 'relu') %>%
  layer_dense(3,activation='softmax')

  neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.001),
    metrics=c('accuracy')
  )

  neuralmodel %>% fit(
  as.matrix(train_data[,-target_indices]),
  as.matrix(train_data[,target_indices]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 30
  )
  
  yhat <- predict(neuralmodel, as.matrix(test_data[-target_indices]))

  #cnn_pred <- cnn_model %>%    TRY THIS FOR RESULT PROCESSING
  #  predict(x_test) %>% k_argmax()

  yhat_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhat)))
  y_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhat)))
  for (i in 1:nrow(yhat)){
    yhat_f[i,]<-which.max(yhat[i,])
    y_f[i,]<-which.max(test_data[i,target_indices])  
  }
  colnames(yhat_f)<-'damage_grade'
  colnames(y_f)<-'damage_grade'

  accuracy_vec[i]<-F1_Score_micro(as.factor(y_f[,]),as.factor(yhat_f[,]))
  setTxtProgressBar(pb, i)
  print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |                                                                      |   0%[1] "F1-Score Micro - 51843 fold: 0.756437706151264"
[1] "F1-Score Micro - 51843 fold: 0.755685434870667"
[1] "F1-Score Micro - 51843 fold: 0.75300426287059"
[1] "F1-Score Micro - 51843 fold: 0.75344791003607"
[1] "F1-Score Micro - 51839 fold: 0.75485638226046"
[1] "Mean F1-Score Micro: NA"


In [13]:
k = 5

nrows<-nrow(dataNN)
split <- floor(nrows*0.8)

# 1. Shuffle the dataset randomly.
dataNN_idx <- sample(1:nrows)


# 2. Split the dataset into k groups
max <- ceiling(nrow(dataNN)/k)
splits <- split(dataNN_idx, ceiling(seq_along(dataNN_idx)/max))


target_indices<-which(grepl('damage_grade',colnames(dataNN)))
accuracy_vec <- array(0,k)


pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

  #3.1 Take the group as a hold out or test data set
  test_data <- dataNN[splits[[i]],]


  #3.2 Take the remaining groups as a training data set
  train_data <- dataNN[-splits[[i]],]   

  normalizer<-layer_normalization(axis = -1L)  %>%  adapt(as.matrix(train_data[,-target_indices]))

  neuralmodel <- keras_model_sequential() %>% 
  normalizer  %>% 
  layer_dense(40, activation = 'relu') %>%
  layer_dense(3,activation='softmax')

  neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.001),
    metrics=c('accuracy')
  )

  neuralmodel %>% fit(
  as.matrix(train_data[,-target_indices]),
  as.matrix(train_data[,target_indices]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 30
  )
  
  yhat <- predict(neuralmodel, as.matrix(test_data[-target_indices]))

  #cnn_pred <- cnn_model %>%    TRY THIS FOR RESULT PROCESSING
  #  predict(x_test) %>% k_argmax()

  yhat_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhat)))
  y_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhat)))
  for (i in 1:nrow(yhat)){
    yhat_f[i,]<-which.max(yhat[i,])
    y_f[i,]<-which.max(test_data[i,target_indices])  
  }
  colnames(yhat_f)<-'damage_grade'
  colnames(y_f)<-'damage_grade'

  accuracy_vec[i]<-F1_Score_micro(as.factor(y_f[,]),as.factor(yhat_f[,]))
  setTxtProgressBar(pb, i)
  print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |                                                                      |   0%[1] "F1-Score Micro - 51843 fold: 0.757903670698069"
[1] "F1-Score Micro - 51843 fold: 0.752849950813032"
[1] "F1-Score Micro - 51843 fold: 0.754470227417395"
[1] "F1-Score Micro - 51843 fold: 0.756283394093706"
[1] "F1-Score Micro - 51839 fold: 0.753409595092498"
[1] "Mean F1-Score Micro: NA"
